# Práctico 2: Curación de Datos e Integración

## Objetivo
Preparar el dataset para el modelado, integrando los tres archivos en una única estructura de análisis coherente.

## 1) Importación de Librerías

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 2) Carga de Datos

In [2]:
company = pd.read_csv('../data/raw/ai_company_adoption.csv')
country = pd.read_csv('../data/raw/country_ai_index.csv')
industry = pd.read_csv('../data/raw/ai_industry_summary.csv')

print('Dimensiones originales:')
print(f'  Company:  {company.shape}')
print(f'  Country:  {country.shape}')
print(f'  Industry: {industry.shape}')

Dimensiones originales:
  Company:  (150025, 43)
  Country:  (30, 8)
  Industry: (9, 8)


## 3) Integración de Archivos

**Estrategia:** Left join manteniendo todos los registros de company como tabla base.

**Justificación:** Company es el nivel de análisis principal. Country e industry son contextuales.

In [ ]:
# `region` aparece en company Y en country. Antes de unir hay que decidir cual
# queda, o pandas las renombra a region_x / region_y y rompe todo lo que siga.
chk = company.merge(country, on='country', how='left', suffixes=('_emp', '_pais'))
discrepan = int((chk['region_emp'] != chk['region_pais']).sum())
ambas_presentes = int(((chk['region_emp'] != chk['region_pais'])
                       & chk['region_emp'].notna()
                       & chk['region_pais'].notna()).sum())
print(f'Chequeo de `region`: {discrepan} discrepancias, '
      f'{ambas_presentes} con ambos valores presentes')
print('  -> es la misma variable. La region es un atributo del pais, asi que')
print('     conservamos la de country  y descartamos la de')
print('     company, que ademas tiene un faltante que country completa.')
print()

# --- Merge 1: company + country (por `country`) ---
df = company.drop(columns=['region']).merge(country, on='country', how='left')
print(f'Merge con country: {df.shape}')
print(f'  sin match: {int(df["gdp_per_capita"].isna().sum())} '
      f'(= filas con `country` nulo)')

# --- Merge 2: + industry (por `industry`) ---
df = df.merge(industry, on='industry', how='left')
print(f'Merge con industry: {df.shape}')
print(f'  sin match: {int(df["avg_ai_adoption_rate"].isna().sum())} '
      f'(= filas con `industry` nulo)')
print()
print('Registros sin match: en ambos casos son exactamente las filas donde la')
print('clave venia vacia, no claves que no existan en las tablas de referencia.')
print('El left join las conserva con NaN en las columnas de contexto; se resuelven')
print('en el paso 4 junto con el resto de los faltantes.')
print()
print(f'Dataset integrado: {df.shape[0]:,} registros x {df.shape[1]} variables')
print('Estructura: cada fila es una respuesta de encuesta de una empresa,')
print('enriquecida con los indicadores de su pais y los promedios de su industria.')

Chequeo de `region`: 2 discrepancias, 0 con ambos valores presentes
  -> es la misma variable. La region es un atributo del pais, asi que
     conservamos la de country (fuente autoritativa) y descartamos la de
     company, que ademas tiene un faltante que country completa.

Merge con country: (150025, 49)
  sin match: 1 (= filas con `country` nulo)
Merge con industry: (150025, 56)
  sin match: 2253 (= filas con `industry` nulo)

Registros sin match: en ambos casos son exactamente las filas donde la
clave venia vacia, no claves que no existan en las tablas de referencia.
El left join las conserva con NaN en las columnas de contexto; se resuelven
en el paso 4 junto con el resto de los faltantes.

Dataset integrado: 150,025 registros x 56 variables
Estructura: cada fila es una respuesta de encuesta de una empresa,
enriquecida con los indicadores de su pais y los promedios de su industria.


## 4) Análisis y Tratamiento de Valores Faltantes

In [4]:
# --- 4.1 Cuanto falta y como se distribuye -------------------------------
miss = pd.DataFrame({
    'NaN': df.isna().sum(),
    '%': (100 * df.isna().mean()).round(2),
}).query('NaN > 0').sort_values('NaN', ascending=False)
print('VARIABLES CON FALTANTES')
print('=' * 60)
print(miss.to_string())
print()

# La pregunta que decide la estrategia: los NaN se apilan en las mismas filas
# o estan repartidos? Cambia por completo el costo de borrar filas.
criticas = ['ai_adoption_rate', 'industry', 'company_size',
            'annual_revenue_usd_millions', 'survey_source', 'country', 'survey_year']
por_fila = df[criticas].isna().sum(axis=1)
print('FALTANTES POR FILA (sobre las 7 variables mas afectadas)')
print('=' * 60)
print(por_fila.value_counts().sort_index().to_string())
print()
print(f'  filas con >=1 faltante : {int((por_fila >= 1).sum()):,} '
      f'({100 * (por_fila >= 1).mean():.2f}%)')
print(f'  filas con >=2          : {int((por_fila >= 2).sum()):,}')
print()
print('CONSECUENCIA: cada columna tiene ~1,5% de NaN, pero casi nunca coinciden')
print('en la misma fila. Borrar toda fila con algun faltante no cuesta 1,5%:')
print(f'cuesta {100 * (por_fila >= 1).mean():.1f}%, porque los porcentajes se suman.')
print('Por eso no aplicamos una regla unica: decidimos variable por variable.')

VARIABLES CON FALTANTES
                                  NaN    %
avg_customer_satisfaction        2253  1.5
avg_productivity_change_percent  2253  1.5
industry                         2253  1.5
avg_jobs_created                 2253  1.5
avg_jobs_displaced               2253  1.5
avg_ai_failure_rate              2253  1.5
avg_ai_maturity_score            2253  1.5
avg_ai_adoption_rate             2253  1.5
survey_source                    2252  1.5
data_privacy_level               2252  1.5
ai_primary_tool                  2252  1.5
customer_satisfaction            2252  1.5
ai_ethics_committee              2251  1.5
company_size                     2251  1.5
ai_budget_percentage             2250  1.5
ai_training_hours                2249  1.5
ai_adoption_rate                 2249  1.5
employee_satisfaction_score      2247  1.5
revenue_growth_percent           2247  1.5
annual_revenue_usd_millions      2247  1.5
innovation_score                    3  0.0
remote_work_percentage        

In [ ]:
# --- 4.2 Estrategia por variable ----------------------------------------
print('EVALUACION DE ESTRATEGIAS')
print('=' * 78)
print('''
ai_adoption_rate  (variable objetivo)
  imputar    -> descartado: imputarlo es fabricar la respuesta que el
                P3 tiene que predecir, sin mendicionar ques la mediana comprimiria la varianza.
  borrar var -> imposible, es el objeto del analisis.
  DECISIÓN: eliminar la fila. Sin target, la observacion no sirve para modelar.

industry, company_size  (categoricas que estructuran el analisis)
  imputar    -> descartado: asignar una industria o un tamaño es inventar a que
                segmento pertenece la empresa, y encima son las variables con las
                que estratificamos las demas imputaciones.
  DECISION: eliminar la fila.

annual_revenue, employee_satisfaction, revenue_growth, customer_satisfaction,
ai_training_hours, ai_budget_percentage  (numericas predictoras)
  borrar filas -> descartado: 6 variables x 1,5% independientes se acumulan.
  borrar var   -> descartado: 1,5% no justifica perder la variable entera.
  DECISION: imputar con la mediana del estrato industry x company_size. La
           mediana resiste la asimetria de estas variables mejor que la media, y
           estratificar respeta que una PyME agricola y una tecnologica grande no
           comparten nivel.

ai_primary_tool, ai_use_case, data_privacy_level, ai_ethics_committee,
survey_source  (categoricas de bajo uso en el modelo)
  DECISION: se conservan con NaN. Las que entran al modelo se codifican en el
           paso 6; las demas no se usan como predictoras (las de metadata).
''')

# --- Implementacion ---
n0 = len(df)
df_clean = df.dropna(subset=['ai_adoption_rate', 'industry', 'company_size']).copy()
print(f'Eliminacion de filas (target y categoricas estructurales)')
print(f'  {n0:,} -> {len(df_clean):,}   ({100 * (n0 - len(df_clean)) / n0:.2f}% eliminado)')

a_imputar = ['annual_revenue_usd_millions', 'employee_satisfaction_score',
             'revenue_growth_percent', 'customer_satisfaction',
             'ai_training_hours', 'ai_budget_percentage']
a_imputar = [c for c in a_imputar if c in df_clean.columns]
print()
print('Imputacion por mediana del estrato (industry x company_size)')
for col in a_imputar:
    n = int(df_clean[col].isna().sum())
    df_clean[col] = df_clean.groupby(['industry', 'company_size'])[col] \
                            .transform(lambda x: x.fillna(x.median()))
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())
    print(f'  {col:32} {n:5d} imputados')

print()
print(f'Dataset tras el tratamiento: {df_clean.shape}')
print(f'Comparacion: borrar toda fila con algun NaN habria dejado '
      f'{int(df.dropna(subset=criticas).shape[0]):,} filas;')
print(f'este esquema conserva {len(df_clean):,} '
      f'(+{len(df_clean) - df.dropna(subset=criticas).shape[0]:,}).')

EVALUACION DE ESTRATEGIAS

ai_adoption_rate  (variable objetivo)
  imputar    -> descartado: imputar el target es fabricar la respuesta que el
                P3 tiene que predecir. Ademas la mediana comprimiria la varianza.
  borrar var -> imposible, es el objeto del analisis.
  ELEGIDA: eliminar la fila. Sin target, la observacion no sirve para modelar.

industry, company_size  (categoricas que estructuran el analisis)
  imputar    -> descartado: asignar una industria o un tamanio es inventar a que
                segmento pertenece la empresa, y encima son las variables con las
                que estratificamos las demas imputaciones.
  ELEGIDA: eliminar la fila.

annual_revenue, employee_satisfaction, revenue_growth, customer_satisfaction,
ai_training_hours, ai_budget_percentage  (numericas predictoras)
  borrar filas -> descartado: 6 variables x 1,5% independientes se acumulan.
  borrar var   -> descartado: 1,5% no justifica perder la variable entera.
  ELEGIDA: imputar con la me

## 5) Outliers: tres problemas distintos

Un valor puede llamar la atencion por tres razones que no se tratan igual:

1. **Es imposible** — una tasa de −138% o un empleado de menos. Lo decide la
   definicion de la variable, no su distribucion.
2. **Esta aislado** — no es que sea grande: es que entre el y el resto de los
   datos no hay *nada*. Un hueco vacio es informacion sobre como se genero el
   dato, no sobre la empresa.
3. **Es la cola** — raro pero continuo con el resto. Una empresa de 19.970
   empleados es rara, no imposible, y no hay hueco que la separe.

Los dos primeros son errores; el tercero es variabilidad legitima. El IQR solo
no distingue ninguno de los tres: sobre distribuciones asimetricas marca ~18% de
la muestra por construccion, y al mismo tiempo deja pasar valores imposibles que
caen dentro de sus bigotes.


In [6]:
# --- 5.1 Validez de dominio ---------------------------------------------
limites = {
    'ai_adoption_rate':        (0, 100,  'tasa en %'),
    'ai_budget_percentage':    (0, 100,  '% del presupuesto'),
    'ai_failure_rate':         (0, 100,  '% de proyectos'),
    'task_automation_rate':    (0, 100,  '% de tareas'),
    'remote_work_percentage':  (0, 100,  '% de dotacion'),
    'cost_reduction_percent':  (None, 100, 'reducir >100% => costo negativo'),
    'num_employees':           (0, None, 'conteo de personas'),
    'jobs_displaced':          (0, None, 'conteo de puestos'),
    'jobs_created':            (0, None, 'conteo de puestos'),
    'reskilled_employees':     (0, None, 'conteo de personas'),
    'years_using_ai':          (0, None, 'anios'),
    'company_age':             (0, None, 'anios'),
    'ai_training_hours':       (0, None, 'horas de capacitacion'),
    'ai_investment_per_employee': (0, None, 'monto invertido'),
    'annual_revenue_usd_millions': (0, None, 'facturacion bruta'),
}

print('5.1  VALORES IMPOSIBLES SEGUN LA DEFINICION DE LA VARIABLE')
print('=' * 78)
dominio = {}
for col, (lo, hi, regla) in limites.items():
    if col not in df_clean.columns: continue
    s = df_clean[col]
    m = pd.Series(False, index=df_clean.index)
    if lo is not None: m |= s < lo
    if hi is not None: m |= s > hi
    if m.any():
        dominio[col] = m
        print(f'  {col:30} {int(m.sum()):5d}   {regla}')
print()
print(f'Variables afectadas: {len(dominio)}')
print('Las demas (num_employees min=10, jobs_* min=0, los % acotados) pasan.')

5.1  VALORES IMPOSIBLES SEGUN LA DEFINICION DE LA VARIABLE
  ai_adoption_rate                 294   tasa en %
  cost_reduction_percent            65   reducir >100% => costo negativo
  ai_training_hours                145   horas de capacitacion
  ai_investment_per_employee       145   monto invertido
  annual_revenue_usd_millions      158   facturacion bruta

Variables afectadas: 5
Las demas (num_employees min=10, jobs_* min=0, los % acotados) pasan.


In [7]:
# --- 5.2 Grupos aislados por un hueco vacio ------------------------------
def bandas_aisladas(s, min_iqr=1.4, max_frac=0.01, min_n=10):
    """Devuelve los grupos que un hueco SIN observaciones separa del cuerpo.
    El hueco se mide en unidades del IQR de la propia variable, y solo se busca
    fuera del rango intercuartil."""
    v = np.sort(s.dropna().unique())
    if len(v) < 50: return {}
    q1, q3 = np.percentile(s.dropna(), [25, 75]); iqr = q3 - q1
    if iqr <= 0: return {}
    d = np.diff(v); res = {}
    for lado in ('sup', 'inf'):
        cand = [(d[i], v[i]) for i in range(len(d))
                if (v[i] >= q3 if lado == 'sup' else v[i + 1] <= q1)]
        if not cand: continue
        ancho, corte = max(cand)
        m = (s > corte) if lado == 'sup' else (s <= corte)
        n = int(m.sum())
        if ancho / iqr >= min_iqr and min_n <= n <= max_frac * s.notna().sum():
            res[lado] = dict(corte=corte, gap=ancho / iqr, n=n, mask=m)
    return res

print('5.2  GRUPOS SEPARADOS DEL CUERPO POR UN HUECO VACIO')
print('=' * 78)
print(f"{'variable':30} {'lado':4} {'hueco':>8} {'n':>5} {'rango del grupo':>22} {'unif?':>6}")
print('-' * 78)

aislados = {}
for col in df_clean.select_dtypes(include=[np.number]).columns:
    if col in ('response_id', 'survey_year', 'company_founding_year'): continue
    for lado, r in bandas_aisladas(df_clean[col]).items():
        g = df_clean.loc[r['mask'], col]
        medio = (g.min() + g.max()) / 2
        # en una uniforme la media coincide con el punto medio del rango;
        # en una cola natural la media queda corrida hacia el cuerpo
        unif = 'SI' if abs(g.mean() - medio) < 0.08 * (g.max() - g.min()) else 'no'
        aislados.setdefault(col, []).append((lado, r, unif))
        print(f"{col:30} {lado:4} {r['gap']:7.1f}x {r['n']:5d} "
              f'{f"[{g.min():,.1f} , {g.max():,.1f}]":>22} {unif:>6}')

print()
print(f'Total de valores en grupos aislados: '
      f"{sum(r['n'] for v in aislados.values() for _, r, _ in v):,}")

5.2  GRUPOS SEPARADOS DEL CUERPO POR UN HUECO VACIO
variable                       lado    hueco     n        rango del grupo  unif?
------------------------------------------------------------------------------
annual_revenue_usd_millions    sup     13.5x   139  [15,538.4 , 29,767.7]     SI
annual_revenue_usd_millions    inf     32.3x   158 [-27,463.8 , -13,220.7]     SI
ai_training_hours              sup      1.5x   148        [106.9 , 183.2]     SI
ai_investment_per_employee     inf      7.6x   145 [-962,812.2 , -456,219.0]     SI
employee_satisfaction_score    sup     80.8x    69        [102.6 , 103.8]     SI
productivity_change_percent    sup      4.4x    77        [110.8 , 119.2]     SI
revenue_growth_percent         inf      3.1x   133        [-58.9 , -27.7]     SI
cost_reduction_percent         sup     12.2x    65        [106.4 , 111.2]     SI
cost_reduction_percent         inf      1.5x    69        [-35.7 , -16.2]     SI

Total de valores en grupos aislados: 1,003


In [8]:
# --- 5.3 Por que 'aislado + uniforme' significa error ---------------------
print('5.3  LA FIRMA DE LOS GRUPOS AISLADOS')
print('=' * 78)
print('''
Cualquier magnitud economica real (facturacion, horas, satisfaccion) se
distribuye con densidad decreciente hacia los extremos: hay muchas empresas
medianas, pocas grandes, poquisimas enormes. Dos consecuencias:

  - No aparecen HUECOS vacios. La densidad baja de a poco, no salta a cero
    para volver a subir mas lejos.
  - La cola NO es uniforme. Su media queda corrida hacia el cuerpo, no en el
    punto medio del rango.

Los grupos de 5.2 violan las dos cosas a la vez. Con eso alcanza: no hace falta
saber como se genero el dato para afirmar que no lo produjo el fenomeno que la
variable dice medir.
''')

for col, lst in aislados.items():
    for lado, r, unif in lst:
        g = df_clean.loc[r['mask'], col]
        cuerpo = df_clean.loc[~r['mask'], col]
        print(f'{col}  ({lado}, n={r["n"]})')
        print(f'  cuerpo         : [{cuerpo.min():>12,.1f} , {cuerpo.max():>12,.1f}]')
        print(f'  hueco vacio    : {r["gap"]:.1f} veces el IQR, sin una sola observacion')
        print(f'  grupo aislado  : [{g.min():>12,.1f} , {g.max():>12,.1f}]')
        print(f'  media del grupo: {g.mean():>12,.1f}   punto medio del rango: '
              f'{(g.min() + g.max()) / 2:>12,.1f}   -> uniforme: {unif}')
        print()

5.3  LA FIRMA DE LOS GRUPOS AISLADOS

Cualquier magnitud economica real (facturacion, horas, satisfaccion) se
distribuye con densidad decreciente hacia los extremos: hay muchas empresas
medianas, pocas grandes, poquisimas enormes. Dos consecuencias:

  - No aparecen HUECOS vacios. La densidad baja de a poco, no salta a cero
    para volver a subir mas lejos.
  - La cola NO es uniforme. Su media queda corrida hacia el cuerpo, no en el
    punto medio del rango.

Los grupos de 5.2 violan las dos cosas a la vez. Con eso alcanza: no hace falta
saber como se genero el dato para afirmar que no lo produjo el fenomeno que la
variable dice medir.

annual_revenue_usd_millions  (sup, n=139)
  cuerpo         : [   -27,463.8 ,      9,996.7]
  hueco vacio    : 13.5 veces el IQR, sin una sola observacion
  grupo aislado  : [    15,538.4 ,     29,767.7]
  media del grupo:     22,954.6   punto medio del rango:     22,653.0   -> uniforme: SI

annual_revenue_usd_millions  (inf, n=158)
  cuerpo         : 

In [ ]:
# --- 5.4 El caso testigo: annual_revenue --------------------------------
# Vale detenerse aca porque la objecion obvia es "una empresa puede tener
# perdidas". Es razonable a priori, y por eso se responde con datos.
s = df_clean['annual_revenue_usd_millions']
tramos = [(-1e9, -13000, 'grupo negativo'), (-13000, 0, 'hueco'),
          (0, 10000, 'CUERPO'), (10000, 13221, 'hueco'),
          (13221, 1e9, 'grupo positivo')]
print('5.4  annual_revenue_usd_millions: donde vive cada cosa')
print('=' * 78)
for lo, hi, nom in tramos:
    n = int(((s >= lo) & (s < hi)).sum())
    print(f'  [{lo:>10,.0f} , {hi:>10,.0f})   {nom:16} {n:>9,}')

neg, pos = s[s < 0].abs(), s[s > 13221]
print()
print('Tres hechos que descartan la lectura de "perdidas":')
print(f'''
1. NO HAY PERDIDAS CHICAS. Si fueran resultados negativos reales, lo mas comun
   serian perdidas modestas. El negativo mas chico en valor absoluto es
   {neg.min():,.0f}, y el 99,9% de las facturaciones positivas esta por debajo de
   {s[(s > 0) & (s < 10000)].quantile(.999):,.0f}. Entre medio: cero observaciones.

2. EL ESPEJO. Del otro lado hay {len(pos)} valores positivos ({pos.min():,.0f} a
   {pos.max():,.0f}) igual de aislados. Un error de signo no explica esos: lo que
   explica a los dos grupos es una sola contaminacion simetrica.

3. SON UNIFORMES. media de |negativos| = {neg.mean():,.0f} contra un punto medio
   del rango de {(neg.min() + neg.max()) / 2:,.0f}. Practicamente iguales: la
   marca de una distribucion uniforme, que ninguna magnitud economica tiene.
''')
tab = pd.DataFrame({
    'observado': df_clean.loc[s < 0, 'company_size'].value_counts(),
    'esperado_si_azar': (df_clean['company_size'].value_counts(normalize=True)
                         * int((s < 0).sum())).round(1)})
print('Ademas no se concentran donde deberian (una perdida de USD 20.000M la')
print('tiene una empresa enorme, no una Startup):')
print(tab.to_string())
print()
print('CONCLUSION: los ~306 valores fuera del cuerpo son contaminacion, no facturacion. El criterio no es el signo: es estar del otro lado del hueco.')

5.4  annual_revenue_usd_millions: donde vive cada cosa
  [-1,000,000,000 ,    -13,000)   grupo negativo         158
  [   -13,000 ,          0)   hueco                    0
  [         0 ,     10,000)   CUERPO             143,070
  [    10,000 ,     13,221)   hueco                    0
  [    13,221 , 1,000,000,000)   grupo positivo         139

Tres hechos que descartan la lectura de "perdidas":

1. NO HAY PERDIDAS CHICAS. Si fueran resultados negativos reales, lo mas comun
   serian perdidas modestas. El negativo mas chico en valor absoluto es
   13,221, y el 99,9% de las facturaciones positivas esta por debajo de
   9,968. Entre medio: cero observaciones.

2. EL ESPEJO. Del otro lado hay 139 valores positivos (15,538 a
   29,768) igual de aislados. Un error de signo no explica esos: lo que
   explica a los dos grupos es una sola contaminacion simetrica.

3. SON UNIFORMES. media de |negativos| = 20,629 contra un punto medio
   del rango de 20,342. Practicamente iguales: la
   marca d

In [11]:
# --- 5.5 Veredicto por variable ------------------------------------------
import textwrap
veredictos = [
 ('annual_revenue_usd_millions', 'ERROR',
  'Dos grupos uniformes fuera del cuerpo [1 , 9.997], separados por huecos sin observaciones. Ni perdidas ni empresas gigantes: contaminacion simetrica.'),
 ('ai_adoption_rate', 'ERROR',
  'Tasa acotada [0,100] con valores de -138 y 210. Es la variable objetivo, asi que no se puede imputar.'),
 ('ai_training_hours', 'ERROR',
  'Horas negativas (imposibles) y un grupo uniforme en [107 , 183] separado del cuerpo, que llega a 80.'),
 ('ai_investment_per_employee', 'ERROR',
  'Montos de inversion negativos, aislados del cuerpo por un hueco de 7,5 IQR.'),
 ('employee_satisfaction_score', 'ERROR',
  'La escala es 0-10 (rango intercuartil 5,0-6,2) y hay 75 casos en 102-104. Es el hueco mas grande de todo el dataset: 79 veces el IQR.'),
 ('productivity_change_percent', 'ERROR',
  'Grupo uniforme en [111 , 119] separado del cuerpo, que termina en 75,5. Antes lo habiamos dado por ambiguo: el hueco vacio lo resuelve.'),
 ('cost_reduction_percent', 'ERROR',
  'Reducciones >100% (costo negativo) y un grupo aislado por abajo. Los dos grupos son uniformes.'),
 ('revenue_growth_percent', 'ERROR PARCIAL',
  'Decrecer es normal: ~30.000 negativos hasta -5,0 son reales. Pero entre -5,0 y -27,7 no hay ninguna observacion, y los 139 valores de mas abajo '
  'son uniformes: esos son contaminacion.'),
 ('num_employees', 'VALIDO',
  'Minimo 10, maximo 19.970, sin huecos: la densidad decrece de forma continua. PyMEs y multinacionales en la misma muestra.'),
 ('jobs_displaced / jobs_created', 'VALIDO',
  'Minimo 0 (con ~9.000 y ~5.500 empresas en cero, plausible) y cola continua que acompania al tamanio de la empresa.'),
]
print('5.5  VEREDICTO')
print('=' * 78)
for var, tipo, just in veredictos:
    print(f'\n[{tipo:13}] {var}')
    for l in textwrap.wrap(just, 66):
        print(f'                 {l}')

5.5  VEREDICTO

[ERROR        ] annual_revenue_usd_millions
                 Dos grupos uniformes fuera del cuerpo [1 , 9.997], separados por
                 huecos sin observaciones. Ni perdidas ni empresas gigantes:
                 contaminacion simetrica.

[ERROR        ] ai_adoption_rate
                 Tasa acotada [0,100] con valores de -138 y 210. Es la variable
                 objetivo, asi que no se puede imputar.

[ERROR        ] ai_training_hours
                 Horas negativas (imposibles) y un grupo uniforme en [107 , 183]
                 separado del cuerpo, que llega a 80.

[ERROR        ] ai_investment_per_employee
                 Montos de inversion negativos, aislados del cuerpo por un hueco de
                 7,5 IQR.

[ERROR        ] employee_satisfaction_score
                 La escala es 0-10 (rango intercuartil 5,0-6,2) y hay 75 casos en
                 102-104. Es el hueco mas grande de todo el dataset: 79 veces el
                 IQR.

[ERROR        

In [ ]:
# --- 5.6 Tratamiento -----------------------------------------------------
# Los errores son independientes entre columnas: casi ninguna fila tiene dos.
# Borrar la fila entera tira variables sanas, asi que se trata la CELDA, salvo
# en el target, que no se puede imputar sin fabricar la respuesta.
df_valid = df_clean.copy()
n0 = len(df_valid)

malos = {}
for col, m in dominio.items():
    malos[col] = m.reindex(df_valid.index, fill_value=False)
for col, lst in aislados.items():
    for _, r, _ in lst:
        m = r['mask'].reindex(df_valid.index, fill_value=False)
        malos[col] = malos.get(col, pd.Series(False, index=df_valid.index)) | m

print('5.6  TRATAMIENTO')
print('=' * 78)
por_fila = pd.DataFrame(malos).sum(axis=1)
print(f'Celdas invalidas: {int(por_fila.sum()):,} en {int((por_fila > 0).sum()):,} filas')
print(f'Filas con 2 o mas: {int((por_fila > 1).sum())}  -> los errores no se apilan')
print()

# (a) target
mt = malos.pop('ai_adoption_rate')
df_valid = df_valid.loc[~mt].copy()
print(f'(a) ai_adoption_rate invalido -> {int(mt.sum())} filas eliminadas ')

# (b) predictoras: a NaN e imputadas por mediana del estrato
print('(b) predictoras -> NaN + mediana de industry x company_size')
for col, m in malos.items():
    m = m.loc[df_valid.index]
    if not m.any(): continue
    df_valid.loc[m, col] = np.nan
    df_valid[col] = (df_valid.groupby(['industry', 'company_size'])[col]
                     .transform(lambda x: x.fillna(x.median())))
    df_valid[col] = df_valid[col].fillna(df_valid[col].median())
    print(f'      {col:32} {int(m.sum()):5d}')

print()
print(f'Filas: {n0:,} -> {len(df_valid):,}  ({100 * (n0 - len(df_valid)) / n0:.3f}% eliminado)')

5.6  TRATAMIENTO
Celdas invalidas: 1,442 en 1,434 filas
Filas con 2 o mas: 8  -> los errores no se apilan

(a) ai_adoption_rate invalido -> 294 filas eliminadas (el target no se imputa)
(b) predictoras -> NaN + mediana de industry x company_size
      cost_reduction_percent             133
      ai_training_hours                  291
      ai_investment_per_employee         145
      annual_revenue_usd_millions        296
      employee_satisfaction_score         69
      productivity_change_percent         77
      revenue_growth_percent             133

Filas: 143,367 -> 143,073  (0.205% eliminado)


In [14]:
# --- 5.7 Lo que queda: cola pesada, no error -----------------------------
print('5.7  POR QUE EL IQR MARCABA 18%')
print('=' * 78)
print(f"{'variable':30} {'asimetria':>10} {'%IQR':>7} {'%|z|>3':>8} {'%IQR log':>9}")
print('-' * 78)
for col in ['annual_revenue_usd_millions', 'num_employees', 'jobs_displaced',
            'jobs_created', 'productivity_change_percent', 'revenue_growth_percent']:
    s = df_valid[col].dropna()
    q1, q3 = s.quantile([.25, .75]); iqr = q3 - q1
    p_iqr = 100 * (((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).mean())
    p_z = 100 * ((np.abs((s - s.mean()) / s.std()) > 3).mean())
    if s.min() > 0:
        ls = np.log(s); a, b = ls.quantile([.25, .75]); li = b - a
        p_log = f'{100 * (((ls < a - 1.5 * li) | (ls > b + 1.5 * li)).mean()):8.2f}'
    else:
        p_log = '     n/a'
    print(f'{col:30} {s.skew():10.2f} {p_iqr:6.2f}% {p_z:7.2f}% {p_log}')
print()
print('''Los tres criterios miden lo mismo sobre los mismos datos. Donde la
asimetria es alta, el IQR marca ~18% y los otros dos menos del 4%: la diferencia
es del metodo, no de los datos. `Q3 + 1,5*IQR` supone simetria, y en una
distribucion con cola derecha larga ese umbral cae muy por debajo del maximo
legitimo. Donde la distribucion es casi simetrica, los tres coinciden.

Estos valores se CONSERVAN: son continuos con el cuerpo, sin huecos, y
corresponden a empresas grandes reales.''')

5.7  POR QUE EL IQR MARCABA 18%
variable                        asimetria    %IQR   %|z|>3  %IQR log
------------------------------------------------------------------------------
annual_revenue_usd_millions          2.27  18.62%    3.41%     0.00
num_employees                        2.28  18.58%    3.46%     0.00
jobs_displaced                       3.04  17.71%    3.22%      n/a
jobs_created                         2.79  18.44%    3.20%      n/a
productivity_change_percent          0.21   0.50%    0.27%      n/a
revenue_growth_percent               0.95   0.51%    0.24%      n/a

Los tres criterios miden lo mismo sobre los mismos datos. Donde la
asimetria es alta, el IQR marca ~18% y los otros dos menos del 4%: la diferencia
es del metodo, no de los datos. `Q3 + 1,5*IQR` supone simetria, y en una
distribucion con cola derecha larga ese umbral cae muy por debajo del maximo
legitimo. Donde la distribucion es casi simetrica, los tres coinciden.

Estos valores se CONSERVAN: son continuos

## 6) Variables categoricas

Tres tratamientos segun lo que la variable representa:

| Tipo | Variables | Criterio |
|---|---|---|
| **Ordinal** | `company_size`, `quarter`, `ai_adoption_stage`, `data_privacy_level`, `country_ai_policy` | tienen un orden real; codificarlas como enteros lo conserva y gasta una sola columna |
| **Binaria** | `ai_ethics_committee` | Si/No -> 1/0 |
| **One-hot** | `industry`, `region` | no hay orden entre rubros ni entre regiones; un entero inventaria que Retail < Technology |

Se descartan `ai_primary_tool`, `ai_use_case` y `company_age_group`: las dos
primeras por alta cardinalidad (se difieren al P3), la tercera por ser una
discretizacion de `company_age`, que ya esta en el dataset como continua.


In [15]:
# --- Mapeos ordinales ----------------------------------------------------
# Cada mapeo se declara con TODAS las categorias observadas. `.map()` devuelve
# NaN silenciosamente ante una categoria no listada, asi que despues validamos.
mapeos = {
    'company_size':       {'Startup': 1, 'SME': 2, 'Enterprise': 3},
    'quarter':            {'Q1': 1, 'Q2': 2, 'Q3': 3, 'Q4': 4},
    'ai_adoption_stage':  {'none': 0, 'pilot': 1, 'partial': 2, 'full': 3},
    'data_privacy_level': {'Low': 1, 'Medium': 2, 'High': 3},
    'country_ai_policy':  {'Lenient': 1, 'Moderate': 2, 'Strict': 3},
}
nuevos = {
    'company_size': 'company_size_enc', 'quarter': 'quarter_enc',
    'ai_adoption_stage': 'ai_stage_enc', 'data_privacy_level': 'privacy_enc',
    'country_ai_policy': 'policy_enc',
}

df_encoded = df_valid.copy()

print('ENCODING ORDINAL')
print('=' * 78)
for col, m in mapeos.items():
    if col not in df_encoded.columns:
        continue
    observadas = set(df_encoded[col].dropna().unique())
    sin_mapear = observadas - set(m)
    assert not sin_mapear, f'{col}: categorias sin mapear -> {sin_mapear}'
    df_encoded[nuevos[col]] = df_encoded[col].map(m)
    # el encoding no puede introducir faltantes nuevos
    assert df_encoded[nuevos[col]].isna().sum() == df_encoded[col].isna().sum()
    orden = ' < '.join(k for k, _ in sorted(m.items(), key=lambda kv: kv[1]))
    print(f'  {col:20} -> {nuevos[col]:20} {orden}')

# --- Binaria -------------------------------------------------------------
print()
print('ENCODING BINARIO')
print('=' * 78)
df_encoded['ethics_enc'] = df_encoded['ai_ethics_committee'].map({'Yes': 1, 'No': 0})
print(f'  ai_ethics_committee  -> ethics_enc            No=0, Yes=1')

# --- One-hot -------------------------------------------------------------
print()
print('ONE-HOT')
print('=' * 78)
for col, pref in [('industry', 'ind'), ('region', 'reg')]:
    dummies = pd.get_dummies(df_encoded[col], prefix=pref, dtype=int)
    df_encoded = pd.concat([df_encoded, dummies], axis=1)
    print(f'  {col:20} -> {len(dummies.columns)} columnas: '
          f'{", ".join(dummies.columns[:3])}, ...')

print()
print(f'Dataset codificado: {df_encoded.shape}')
print()
print('VALIDACION -- las codificadas no pueden tener mas NaN que su origen:')
for orig, nuevo in list(nuevos.items()) + [('ai_ethics_committee', 'ethics_enc')]:
    if orig in df_encoded.columns:
        a, b = int(df_encoded[orig].isna().sum()), int(df_encoded[nuevo].isna().sum())
        print(f'  {nuevo:20} NaN origen={a:5d}  NaN codificada={b:5d}  '
              f"{'OK' if a == b else 'ERROR'}")

ENCODING ORDINAL
  company_size         -> company_size_enc     Startup < SME < Enterprise
  quarter              -> quarter_enc          Q1 < Q2 < Q3 < Q4
  ai_adoption_stage    -> ai_stage_enc         none < pilot < partial < full
  data_privacy_level   -> privacy_enc          Low < Medium < High
  country_ai_policy    -> policy_enc           Lenient < Moderate < Strict

ENCODING BINARIO
  ai_ethics_committee  -> ethics_enc            No=0, Yes=1

ONE-HOT
  industry             -> 9 columnas: ind_Agriculture, ind_Consulting, ind_Education, ...
  region               -> 6 columnas: reg_Africa, reg_Asia, reg_Europe, ...

Dataset codificado: (143073, 77)

VALIDACION -- las codificadas no pueden tener mas NaN que su origen:
  company_size_enc     NaN origen=    0  NaN codificada=    0  OK
  quarter_enc          NaN origen=    0  NaN codificada=    0  OK
  ai_stage_enc         NaN origen=    1  NaN codificada=    1  OK
  privacy_enc          NaN origen= 2149  NaN codificada= 2149  OK
  po

## 7) Selección de Variables para Modelización

Las decisiones de curacion no se toman en el vacio: responden a las preguntas que
el P3 tiene que poder responder. En el EDA planteamos tres hipotesis; aca se
explicita que hicimos con las variables que cada una necesita.


**H1: ¿La inversion en IA mejora la productividad, controlando por industria y tamaño?**

Las variables centrales son `ai_investment_per_employee` y `productivity_change_percent`,
con `industry` y `company_size` como controles. `ai_investment_per_employee` tenia
145 valores negativos —imposibles para un monto de inversion— que se trataron como
errores y se imputaron por mediana del estrato. `productivity_change_percent` presentaba
un grupo de 77 valores en [111, 119] separado del cuerpo por un hueco de 4,4 veces el
IQR y distribuido de forma uniforme: la firma de contaminacion, no de empresas
extraordinariamente productivas. Se convirtieron a NaN y se imputaron. Las variables
de control `industry` y `company_size` no se imputaron: si faltaban, se elimino la fila,
porque fabricar un segmento seria inventar el estrato que el modelo tiene que controlar.

**H2: ¿Las diferencias de adopcion entre industrias se mantienen al controlar tamaño y pais?**

La variable objetivo es `ai_adoption_rate`, y los controles son `industry`, `company_size`
y el contexto geografico. `industry` se codifico como one-hot para que cada sector tenga
su propio coeficiente en el modelo, que es exactamente lo que la pregunta pide estimar.
La integracion con la tabla `country_ai_index` agrego indicadores de desarrollo
(`gdp_per_capita`, `digital_maturity_index`, `internet_penetration`) que permiten
controlar el contexto del pais mas alla de su nombre. Las columnas `avg_*` se excluyeron
porque son promedios por industria calculados sobre la propia tabla: incluir
`avg_ai_adoption_rate` como predictora de `ai_adoption_rate` seria responder la pregunta
con informacion circular.


**H3: ¿La adopcion de IA destruye empleo neto o transforma puestos de trabajo?**

Las variables de interes son `jobs_created`, `jobs_displaced` y `reskilled_employees`,
junto con `ai_adoption_rate` y `task_automation_rate` como explicativas. Las tres
variables de empleo se verificaron: tienen minimo 0 (plausible), sin valores negativos,
y sus colas son continuas con el cuerpo —sin huecos que sugieran contaminacion—. Se
conservaron integramente. El analisis de correlaciones mostro que las tres estan
altamente correlacionadas entre si y con `num_employees` (r ~ 0,90): no son tres
fenomenos distintos, sino el mismo tamaño de empresa medido de distintas formas. Para
responder si la IA destruye empleo *en proporcion al tamaño*, el P3 deberia normalizar
(por ejemplo, `jobs_displaced / num_employees`) en lugar de usar los conteos absolutos.


In [ ]:
print('SELECCION DE VARIABLES PARA EL P3')
print('=' * 78)
print()
# La tabla industry no aporta datos de otra fuente: sus columnas son agregados
# calculados sobre la propia tabla company. Se verifica:
verif = pd.concat([
    company.groupby('industry')['ai_adoption_rate'].mean().rename('recalculado'),
    industry.set_index('industry')['avg_ai_adoption_rate'].rename('en el csv'),
], axis=1).round(2)
print(verif.to_string())
print()
print('Coinciden hasta el segundo decimal: `avg_ai_adoption_rate` ES el promedio por industria de nuestra variable objetivo.')
print('En el P3 daria un ajuste excelente y enganoso.')
print()
print('Ademas, TODAS las `avg_*` son constantes dentro de cada industria, o sea')
print('combinacion lineal exacta de las dummies `ind_*`. Tenerlas junto al one-hot')
print('de industria introduce colinealidad perfecta.')
print()
print('DECISION: se excluyen las 7 columnas `avg_*`. La informacion de industria')
print('entra por las dummies `ind_*`, que la capturan sin circularidad.')
print()
print('--- Otras exclusiones ---')
print('  response_id, company_id ....... identificadores, sin contenido predictivo')
print('  survey_year, quarter, source .. metadatos del relevamiento, no de la empresa')
print('  ai_primary_tool, ai_use_case .. alta cardinalidad; se difieren al P3')
print('  company_age_group ............. discretizacion de company_age (redundante)')
print('  industry, region, company_size  las originales, ya codificadas')
print()

keep_vars = [
    # objetivo
    'ai_adoption_rate',
    # perfil de la empresa
    'company_size_enc', 'num_employees', 'annual_revenue_usd_millions', 'company_age',
    # capacidad instalada en IA
    'years_using_ai', 'ai_stage_enc', 'ai_maturity_score', 'num_ai_tools_used',
    'ai_projects_active',
    # inversion
    'ai_training_hours', 'ai_budget_percentage', 'ai_investment_per_employee',
    # gobernanza
    'ethics_enc', 'privacy_enc', 'ai_risk_management_score',
    'regulatory_compliance_score',
    # resultados
    'productivity_change_percent', 'task_automation_rate', 'time_saved_per_week',
    'jobs_displaced', 'jobs_created', 'reskilled_employees',
    'revenue_growth_percent', 'cost_reduction_percent', 'innovation_score',
    'customer_satisfaction', 'employee_satisfaction_score', 'ai_failure_rate',
    # contexto pais
    'gdp_per_capita', 'internet_penetration', 'digital_maturity_index',
    'ai_patent_filings_2024', 'ai_researchers_per_million', 'policy_enc',
]
keep_vars += [c for c in df_encoded.columns if c.startswith(('ind_', 'reg_'))]
keep_vars = [v for v in keep_vars if v in df_encoded.columns]

fuga = [c for c in keep_vars if c.startswith('avg_')]
assert not fuga, f'quedaron columnas con fuga: {fuga}'

df_model = df_encoded[keep_vars].copy()
print(f'Variables seleccionadas: {len(keep_vars)}')
print(f'Dataset para el P3: {df_model.shape[0]:,} x {df_model.shape[1]}')
print()

# Redundancias entre las que quedaron
num = df_model.select_dtypes(include=[np.number])
cm = num.corr().abs()
pares = [(cm.columns[i], cm.columns[j], cm.iloc[i, j])
         for i in range(len(cm)) for j in range(i + 1, len(cm))
         if cm.iloc[i, j] > 0.8]
print('Pares con |r| > 0,8 entre las variables retenidas:')
if pares:
    for a, b, r in sorted(pares, key=lambda x: -x[2]):
        print(f'  {a} <-> {b}: {r:.3f}')
else:
    print('  ninguno')

print()
print('LECTURA DE LAS REDUNDANCIAS')
print('=' * 78)
print('''
Los pares se agrupan en tres familias, y cada una pide una decision distinta.

1) ESCALA DE LA EMPRESA
   num_employees ~ jobs_created ~ jobs_displaced ~ reskilled_employees (r ~ 0,90)
   No son cuatro fenomenos: son el mismo tamanio medido cuatro veces. Se
   conservan las cuatro en el dataset, pero se deja anotado para el P3 que
   conviene normalizarlas por empleado (jobs_created / num_employees) o quedarse
   con una sola, segun el modelo que se use.

2) DESARROLLO DEL PAIS
   gdp_per_capita ~ internet_penetration ~ digital_maturity_index ~
   ai_researchers_per_million (r ~ 0,85-0,95)
   Cuatro indicadores del mismo constructo latente, y solo 30 paises distintos:
   aportan poca variacion propia. Candidatas naturales a colapsar en un solo
   indice en el P3.

3) *** El caso a mirar con cuidado ***
   ai_adoption_rate ~ ai_stage_enc      r = 0,86
   ai_adoption_rate ~ ai_maturity_score r = 0,84
   `ai_adoption_stage` (none/pilot/partial/full) no es una causa de la adopcion:
   es la MISMA variable objetivo contada en categorias. Usarla como predictora
   es explicar la tasa con una version discretizada de si misma. Lo mismo aplica,
   de forma mas atenuada, a `ai_maturity_score`.

   Se conservan en el dataset porque pueden ser el target de otra pregunta (por
   ejemplo, clasificar la etapa en lugar de predecir la tasa), pero quedan
   MARCADAS: si en el P3 se modela `ai_adoption_rate`, ninguna de las dos puede
   entrar como predictora.
''')

SELECCION DE VARIABLES PARA EL P3

--- Riesgo detectado: las columnas `avg_*` no son informacion externa ---

               recalculado  en el csv
industry                             
Agriculture          35.22      35.25
Consulting           34.57      34.52
Education            35.33      35.23
Finance              38.33      38.35
Healthcare           34.57      34.57
Logistics            35.30      35.26
Manufacturing        34.76      34.73
Retail               34.73      34.70
Technology           42.50      42.47

Coinciden hasta el segundo decimal: `avg_ai_adoption_rate` ES el promediopor industria de nuestra variable objetivo. Incluirla como predictora
significa predecir el target con una funcion del target -> fuga (leakage).
En el P3 daria un ajuste excelente y enganoso.

Ademas, TODAS las `avg_*` son constantes dentro de cada industria, o sea
combinacion lineal exacta de las dummies `ind_*`. Tenerlas junto al one-hot
de industria introduce colinealidad perfecta.

DECISION:

### 7.1 Cierre: dejar el dataset sin faltantes

Quedan dos residuos que el P3 no deberia tener que resolver.


In [ ]:
# --- Residuo 1: categoricas con ~1,5% sin respuesta ----------------------
# `ai_ethics_committee` y `data_privacy_level` no fueron contestadas en ~2.100
# casos. Imputar 'No' o 'Low' seria afirmar algo que el relevamiento no dice, y
# el no-responde puede ser informativo (una empresa sin comite de etica quiza
# tiende a saltear la pregunta). Se codifica el faltante como senal propia.
for enc, flag in [('ethics_enc', 'ethics_na'), ('privacy_enc', 'privacy_na')]:
    n = int(df_model[enc].isna().sum())
    df_model[flag] = df_model[enc].isna().astype(int)
    df_model[enc] = df_model[enc].fillna(df_model[enc].mode()[0])
    print(f'{enc:14} {n:5d} faltantes -> imputados con la moda '
          f'+ indicador `{flag}` que preserva el no-responde')

# --- Residuo 2: filas con nulos dispersos --------------------------------
# Una fila sin `country` arrastra en NaN las 6 columnas de contexto pais; otras
# pocas tienen un nulo suelto heredado del archivo original.
n0 = len(df_model)
df_model = df_model.dropna()
print(f'\nFilas con nulos sueltos eliminadas: {n0 - len(df_model)} '
      f'({100 * (n0 - len(df_model)) / n0:.4f}%)')

# --- Validacion final ----------------------------------------------------
print()
print('VALIDACION DEL DATASET FINAL')
print('=' * 78)
reglas = {
    'ai_adoption_rate': (0, 100), 'annual_revenue_usd_millions': (0, None),
    'num_employees': (0, None), 'jobs_displaced': (0, None),
    'jobs_created': (0, None), 'cost_reduction_percent': (None, 100),
    'ai_budget_percentage': (0, 100), 'task_automation_rate': (0, 100),
}
malos = 0
for c, (lo, hi) in reglas.items():
    if c in df_model.columns:
        if lo is not None:
            malos += int((df_model[c] < lo).sum())
        if hi is not None:
            malos += int((df_model[c] > hi).sum())

checks = [
    ('sin faltantes',            int(df_model.isna().sum().sum()) == 0),
    ('sin valores imposibles',   malos == 0),
    ('sin columnas constantes',  not [c for c in df_model.columns
                                      if df_model[c].nunique() <= 1]),
    ('sin columnas vacias',      not [c for c in df_model.columns
                                      if df_model[c].isna().all()]),
    ('sin fuga (`avg_*`)',       not [c for c in df_model.columns
                                      if c.startswith('avg_')]),
    ('todo numerico',            df_model.select_dtypes(exclude=[np.number]).empty),
]
for nombre, ok in checks:
    print(f"  [{'OK' if ok else 'FALLA'}] {nombre}")
assert all(ok for _, ok in checks), 'el dataset final no pasa la validacion'

print()
print(f'Dataset final: {df_model.shape[0]:,} registros x {df_model.shape[1]} variables')

## 8) Conclusiones Parciales

### 1. Integracion

150.025 × 43 → left join con country (30) e industry (9) → 150.025 × 56

Claves: `country` e `industry`. Left join porque la unidad de analisis es la empresa
y no queremos perder ninguna. `region` estaba en las dos tablas: se conserva la de
country, que es la fuente autoritativa. Los registros sin match son exactamente
aquellos con la clave vacia, no claves inexistentes en las tablas de referencia.

### 2. Valores faltantes

150.025 → 143.367 (4,44% eliminado)

~1,5% por columna, pero casi nunca en la misma fila: borrar toda fila incompleta
costaba 5,9%. Por eso se decidio variable por variable:

- **target nulo** → elimina fila (imputarlo es fabricar la respuesta)
- **industry / company_size** → elimina fila (definen el estrato de imputacion)
- **numericas predictoras** → mediana de industry × company_size
- **no-respuesta en categoricas** → moda + indicador que preserva la senal

### 3. Outliers

143.367 → 143.073 (0,205% eliminado)

Se separaron tres problemas que el IQR mezcla:

**a) Imposibles por definicion** (807 valores en 5 variables)  
Tasas fuera de [0,100], horas y montos negativos, reducciones >100%.

**b) Aislados por un hueco vacio** (1.003 valores en 7 variables)  
El hallazgo principal. En 8 variables hay grupos de 65-160 valores separados del
cuerpo por un intervalo sin una sola observacion, y distribuidos de forma UNIFORME
dentro de su banda. Ninguna magnitud economica se comporta asi: las colas reales se
adelgazan y no dejan huecos. Son contaminacion sintetica, no empresas extremas.  
El caso mas claro: `employee_satisfaction_score` es una escala 0-10 y tiene 69 casos
en 102-104, al otro lado de un hueco de 80 veces el IQR.

**c) Cola pesada → se conservan**  
`annual_revenue`, `num_employees` y `jobs_*` tienen asimetria 1,7-3,0. El IQR marcaba
~18% de la muestra; |z|>3 y el IQR en escala log, menos del 4%. Ese 18% era artefacto
de aplicar un criterio simetrico a datos que no lo son, no una medida de cuantos
datos estaban mal.

**Tratamiento:** el target invalido elimina la fila; las predictoras se llevan a NaN
y se imputan por estrato. Los errores casi nunca coinciden en la misma fila, asi que
borrarlas enteras habria costado varias variables sanas por cada celda corrupta.

### 4. Variables categoricas

- **Ordinales:** `company_size` (Startup < SME < Enterprise), `ai_adoption_stage`
  (none < pilot < partial < full), `data_privacy_level`, `country_ai_policy`, `quarter`.
- **Binaria:** `ai_ethics_committee`.
- **Nominales:** `industry` y `region` → one-hot.

Los mapeos se validan con assert: `.map()` devuelve NaN ante una categoria no listada,
sin avisar, y vaciaria la columna entera en silencio.

### 5. Seleccion de variables

**Dataset final: 143.073 × 50**

Se excluyen IDs, metadata de encuesta y las categoricas ya codificadas. La exclusion
de fondo son las 7 columnas `avg_*`: no son informacion externa, son promedios
calculados sobre la propia tabla company. `avg_ai_adoption_rate` es la media por
industria del target → fuga. Ademas son constantes dentro de cada industria, o sea
colineales con `ind_*`.

**Marcadas (no excluidas):** `ai_stage_enc` (r = 0,86 con el target) y
`ai_maturity_score` (r = 0,84) son el mismo fenomeno medido de otra forma. Si en el
P3 se modela `ai_adoption_rate`, no pueden entrar como predictoras.

---

### Lo mas dificil

Decidir cuando un valor raro es un error. Tres criterios sucesivos dieron tres
respuestas distintas sobre los mismos datos:

| Criterio | Resultado |
|----------|--------|
| **IQR solo** | 18% de outliers en facturacion, y ninguna alarma sobre una tasa de adopcion del 210%. |
| **Validez de dominio** | Encuentra lo imposible, pero deja pasar 69 satisfacciones de 103 sobre una escala de 10 si uno no conocia el tope de la escala. |
| **Hueco + uniformidad** | No necesita conocer la escala ni el signo esperado: se apoya en la FORMA de la distribucion. Encontro 8 variables contaminadas, cinco de las cuales los otros dos no veian. |

El caso testigo es el signo. `revenue_growth_percent` negativo es valido: una empresa
puede decrecer. `annual_revenue` negativo no lo es: la facturacion bruta no tiene
signo. Y dentro de `revenue_growth` conviven las dos cosas: los ~30.000 negativos
hasta -5,0 son empresas que se achicaron, y los 139 que estan entre -27,7 y -58,9 son
contaminacion, porque entre -5,0 y -27,7 no hay una sola observacion. El criterio
nunca fue el signo ni la magnitud: fue si el valor es continuo con el resto.


In [18]:
print('EXPORTACION')
print('=' * 78)
df_encoded.to_csv('../data/raw/data_curada_completa.csv', index=False)
df_model.to_csv('../data/raw/data_modelo.csv', index=False)
print(f'  data_curada_completa.csv  {df_encoded.shape[0]:,} x {df_encoded.shape[1]}'
      '   (todo lo curado, incluidas las columnas no usadas en el modelo)')
print(f'  data_modelo.csv           {df_model.shape[0]:,} x {df_model.shape[1]}'
      '   <- entrada del P3')

EXPORTACION
  data_curada_completa.csv  143,073 x 77   (todo lo curado, incluidas las columnas no usadas en el modelo)
  data_modelo.csv           143,073 x 50   <- entrada del P3
